# Chapter 3 Practical 07: Graph-Based and Explainable Collaborative Filtering

Learning objectives:
- Build a bipartite user-item graph.
- Run Personalized PageRank from a target user.
- Recommend unseen items from graph scores.
- Explain recommendations using paths and neighbor evidence.
- Add a simple hybrid fallback for cold start.

Slide connection: bipartite graphs, Personalized PageRank, graph-based CF benefits, explainable CF, and cold-start fallback.


In [1]:
# Teaching note: Load the rating matrix and movie metadata for graph-based CF.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "ratings_chapter3.csv").exists():
    DATA_DIR = Path("chapter_03_collaborative_filtering/data")

ratings = pd.read_csv(DATA_DIR / "ratings_chapter3.csv")
movies = pd.read_csv(DATA_DIR / "movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
# Pivot interactions into rows = users and columns = items.
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


In [2]:
# Teaching note: Build a bipartite graph with user nodes, item nodes, and rating edges.
import networkx as nx

# A graph stores users/items as nodes and interactions as edges.
G = nx.Graph()
for user in ratings_named["user_id"].unique():
    G.add_node(f"user:{user}", kind="user", label=user)
for _, row in movies.iterrows():
    G.add_node(f"item:{row['title']}", kind="item", label=row["title"], genre=row["genre"])
for _, row in ratings_named.iterrows():
    if row["rating"] >= 4:
        G.add_edge(f"user:{row['user_id']}", f"item:{row['title']}", weight=row["rating"])

print(nx.info(G) if hasattr(nx, "info") else f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


18 nodes, 28 edges


In [3]:
# Teaching note: Run Personalized PageRank from a target user and rank unseen item nodes.
target_user = "Karen"
seed = {node: 0 for node in G.nodes}
seed[f"user:{target_user}"] = 1

# Personalized PageRank spreads relevance through the interaction graph.
ppr = nx.pagerank(G, alpha=0.85, personalization=seed, weight="weight")
seen = set(rating_matrix.loc[target_user].dropna().index)

recommendations = []
for node, score in ppr.items():
    if node.startswith("item:"):
        title = node.removeprefix("item:")
        if title not in seen:
            recommendations.append({"recommended_movie": title, "ppr_score": score})

pd.DataFrame(recommendations).sort_values("ppr_score", ascending=False).head(5)


,recommended_movie,ppr_score
0,Independence Day,0.044488
1,Toy Story,0.027657
4,Blade Runner,0.012845
5,Finding Nemo,0.007530
2,Titanic,0.000002


In [4]:
# Teaching note: Explain a recommendation by listing short paths from the user to the item.
# Graph paths provide a human-readable explanation for PPR recommendations.
def explain_graph_recommendation(graph, user, item, max_paths=3):
    source = f"user:{user}"
    target = f"item:{item}"
    paths = []
    for path in nx.all_simple_paths(graph, source, target, cutoff=4):
        paths.append(" -> ".join(node.replace("user:", "").replace("item:", "") for node in path))
        if len(paths) >= max_paths:
            break
    return paths

explain_graph_recommendation(G, "Karen", "Blade Runner")


['Karen -> Star Wars -> Alice -> Blade Runner',
 'Karen -> The Matrix -> Alice -> Blade Runner']

In [5]:
# Teaching note: Use graph CF when history exists and a popularity fallback for cold-start users.
# Hybrid fallback chooses CF when history exists and popularity when it does not.
def hybrid_recommend(user, n=5):
    if user in rating_matrix.index and rating_matrix.loc[user].notna().sum() >= 2:
        seed = {node: 0 for node in G.nodes}
        seed[f"user:{user}"] = 1
        # Personalized PageRank spreads relevance through the interaction graph.
        scores = nx.pagerank(G, alpha=0.85, personalization=seed, weight="weight")
        seen = set(rating_matrix.loc[user].dropna().index)
        rows = []
        for node, score in scores.items():
            if node.startswith("item:"):
                title = node.removeprefix("item:")
                if title not in seen:
                    rows.append({"movie": title, "score": score, "source": "graph_cf"})
        return pd.DataFrame(rows).sort_values("score", ascending=False).head(n)

    # Groupby aggregates ratings by user or item for summary statistics.
    popular = ratings_named.groupby("title")["rating"].agg(["count", "mean"])
    popular["score"] = popular["mean"] * np.log1p(popular["count"])
    return popular.sort_values("score", ascending=False).head(n).reset_index().assign(source="popular_fallback")

hybrid_recommend("NewStudent")


,title,count,mean,score,source
0,Star Wars,6,5.333333,10.378187,popular_fallback
1,The Matrix,4,6.000000,9.656627,popular_fallback
2,Jurassic Park,7,4.571429,9.506018,popular_fallback
3,Terminator 2,5,4.600000,8.242094,popular_fallback
4,Independence Day,5,4.200000,7.525390,popular_fallback


Exercises:
1. Add genre nodes to the graph and connect movies to genres.
2. Compare graph recommendations with item-item recommendations for Karen.
